In [1]:
import torch
import torch.nn as nn

# Utilities for handling variable-length sequences
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)

class RobustBiLSTM(nn.Module):
    """
    Bidirectional LSTM model for sequence classification.

    Architecture:
    Embedding -> Dropout -> BiLSTM -> Concatenation (Forward + Backward)
    -> Dropout -> Linear Classifier
    """

    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        num_layers,
        num_classes,
        dropout=0.3,
        pad_idx=0
    ):
        """
        Args:
        vocab_size (int): Size of vocabulary
        embed_dim (int): Dimension of embedding vectors
        hidden_dim (int): Hidden size of LSTM
        num_layers (int): Number of stacked LSTM layers
        num_classes (int): Output classes
        dropout (float): Dropout probability
        pad_idx (int): Padding token index
        """
        super().__init__()

        # ----------------------------------------------------
        # Embedding Layer
        # Converts token indices to dense vectors
        # ----------------------------------------------------
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        # ----------------------------------------------------
        # Bidirectional LSTM
        # Processes sequences in both forward and backward directions
        # ----------------------------------------------------
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

        # Final Classification Layer
        # hidden_dim * 2 = bidirectional
        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x, lengths, hidden=None):
        """
        Forward pass.

        Args:
        x (Tensor): Input token indices (batch_size, seq_len)
        lengths (Tensor): Actual lengths of sequences (before padding)
        hidden (tuple, optional): Initial hidden and cell states

        Returns:
        logits (Tensor): Classification scores
        features (Tensor): Final extracted features
        (h_n, c_n): Final hidden and cell states
        unpacked_out (Tensor): Full sequence outputs
        """

        # ----------------------------------------------------
        # 1) Embedding lookup
        # ----------------------------------------------------
        x = self.embedding(x)

        # Apply dropout on embeddings
        x = self.dropout(x)

        # pack_padded_sequence requires CPU lengths
        lengths_cpu = lengths.to("cpu")

        # ----------------------------------------------------
        # 2) Pack padded sequences
        # Efficient computation without padded tokens
        # ----------------------------------------------------
        packed = pack_padded_sequence(
            x,
            lengths_cpu,
            batch_first=True,
            enforce_sorted=False
        )

        # ----------------------------------------------------
        # 3) LSTM Forward Pass
        # ----------------------------------------------------
        packed_out, (h_n, c_n) = self.lstm(
            packed,
            hidden
        )

        # ----------------------------------------------------
        # 4) Unpack sequences back to padded format
        # ----------------------------------------------------
        unpacked_out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )

        # ----------------------------------------------------
        # 5) Extract Final Hidden States
        # h_n shape:
        # (num_layers * num_directions, batch_size, hidden_dim)
        #
        # For BiLSTM:
        # -2 = Last layer forward
        # -1 = Last layer backward
        # ----------------------------------------------------
        forward_last = h_n[-2]
        backward_last = h_n[-1]

        # Concatenate forward and backward representations
        features = torch.cat(
            [forward_last, backward_last],
            dim=1
        )

        # Regularization
        features = self.dropout(features)

        # ----------------------------------------------------
        # 6) Classification Layer
        # ----------------------------------------------------
        logits = self.classifier(features)

        return logits, features, (h_n, c_n), unpacked_out


In [2]:
from torch.nn.utils.rnn import pad_sequence
import torch

def collate_fn(batch, pad_value=0):
    """
    Custom collate function for PyTorch DataLoader.

    Purpose:
    Handles variable-length sequences by padding them to the same
    length within a batch while also returning the original lengths.
    This is required for efficient processing with RNN/LSTM models
    using pack_padded_sequence.

    Args:
    batch (list): List of (sequence, label) tuples
    pad_value (int): Padding value used for shorter sequences

    Returns:
    padded_sequences (Tensor): Shape (batch_size, max_seq_len)
    lengths (Tensor): Original sequence lengths
    labels (Tensor): Target labels
    """

    # ----------------------------------------------------
    # 1) Separate sequences and labels
    # batch = [(seq1, label1), (seq2, label2), ...]
    # ----------------------------------------------------
    sequences, labels = zip(*batch)

    # ----------------------------------------------------
    # 2) Compute original sequence lengths
    # Needed later for pack_padded_sequence
    # ----------------------------------------------------
    lengths = torch.tensor(
        [len(seq) for seq in sequences],
        dtype=torch.long
    )

    # ----------------------------------------------------
    # 3) Pad sequences to equal length
    # Resulting shape: (batch_size, max_seq_len)
    # ----------------------------------------------------
    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=pad_value
    )

    # ----------------------------------------------------
    # 4) Convert labels to tensor
    # ----------------------------------------------------
    labels = torch.tensor(
        labels,
        dtype=torch.long
    )

    # ----------------------------------------------------
    # Return processed batch
    # ----------------------------------------------------
    return padded_sequences, lengths, labels


In [3]:
import random
import torch

from torch.utils.data import Dataset


class ToySequenceDataset(Dataset):
    """
    A simple synthetic dataset for sequence classification.

    Each sample consists of:
    - A randomly generated integer sequence
    - A binary label derived from the sequence

    Label rule:
    If the sum of tokens in the sequence is even -> label = 1
    Otherwise -> label = 0
    """

    def __init__(
        self,
        n_samples=256,
        vocab_size=50,
        min_len=5,
        max_len=20
    ):
        """
        Args:
        n_samples (int): Number of samples in the dataset
        vocab_size (int): Size of the token vocabulary
        min_len (int): Minimum sequence length
        max_len (int): Maximum sequence length
        """

        # Container for all generated samples
        self.samples = []

        # ----------------------------------------------------
        # Generate synthetic samples
        # ----------------------------------------------------
        for _ in range(n_samples):

            # Randomly choose sequence length
            length = random.randint(
                min_len,
                max_len
            )

            # Generate random token sequence
            # Tokens range from 1 to vocab_size-1
            seq = torch.randint(
                1,
                vocab_size,
                (length,),
                dtype=torch.long
            )

            # ----------------------------------------------------
            # Define label based on sequence property
            # Label = 1 if sum of tokens is even
            # ----------------------------------------------------
            label = int(
                seq.sum().item() % 2 == 0
            )

            # Store sample as (sequence, label)
            self.samples.append(
                (seq, label)
            )

    def __len__(self):
        """
        Returns total number of samples.
        Required by PyTorch Dataset interface.
        """
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Retrieves a single sample by index.

        Args:
        idx (int): Sample index

        Returns:
        (sequence, label)
        """
        return self.samples[idx]


In [4]:
import torch.nn as nn


def init_lstm_orthogonal(model):
    """
    Applies custom initialization to LSTM parameters.

    Initialization strategy:
    - Input-to-hidden weights -> Xavier uniform initialization
    - Hidden-to-hidden weights -> Orthogonal initialization
    - Bias terms -> Zero initialization
      with forget gate bias set to 1.0

    This setup is commonly used to improve training stability
    and help recurrent networks preserve information more effectively.
    """

    # Iterate through all named parameters in the model
    for name, param in model.named_parameters():

        # ----------------------------------------------------
        # Initialize input-to-hidden weights
        # ----------------------------------------------------
        if "weight_ih" in name:

            nn.init.xavier_uniform_(
                param.data
            )

        # ----------------------------------------------------
        # Initialize hidden-to-hidden weights
        # Each gate block is initialized orthogonally
        # ----------------------------------------------------
        elif "weight_hh" in name:

            hidden_dim = param.shape[1]

            for start in range(
                0,
                param.shape[0],
                hidden_dim
            ):
                nn.init.orthogonal_(
                    param.data[
                        start:start + hidden_dim
                    ]
                )

        # ----------------------------------------------------
        # Initialize biases
        # ----------------------------------------------------
        elif "bias" in name:

            # Set all bias values to zero
            nn.init.zeros_(param.data)

            # LSTM bias layout is typically:
            # [input_gate | forget_gate | cell_gate | output_gate]
            hidden_dim = param.shape[0] // 4

            # Set forget gate bias to 1.0
            # This helps the model retain information early in training
            param.data[
                hidden_dim:2 * hidden_dim
            ].fill_(1.0)


In [5]:
def detach_state(state):

    if state is None:
        return None

    if isinstance(state, tuple):

        return tuple(
            s.detach() for s in state
        )

    return state.detach()


In [6]:
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from torch.nn.utils import clip_grad_norm_


# ----------------------------------------------------
# Device configuration
# ----------------------------------------------------
device = torch.device("cpu")


# ----------------------------------------------------
# Model hyperparameters
# ----------------------------------------------------
vocab_size = 50
embed_dim = 32
hidden_dim = 64
num_layers = 2
num_classes = 2

batch_size = 16

epochs = 5

learning_rate = 1e-3

# Maximum gradient norm for gradient clipping
max_norm = 1.0


# ----------------------------------------------------
# Dataset and DataLoader
# ----------------------------------------------------
train_dataset = ToySequenceDataset(
    n_samples=256,
    vocab_size=vocab_size
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)


# ----------------------------------------------------
# Model initialization
# ----------------------------------------------------
model = RobustBiLSTM(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    num_classes=num_classes,
    dropout=0.3
).to(device)

# Apply orthogonal initialization to LSTM parameters
init_lstm_orthogonal(model)


# ----------------------------------------------------
# Loss function and optimizer
# ----------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)


# ----------------------------------------------------
# Containers for training history
# ----------------------------------------------------
history_loss = []
history_acc = []


# ----------------------------------------------------
# Training loop
# ----------------------------------------------------
for epoch in range(epochs):

    # Set model to training mode
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    # Initial hidden state (optional stateful training)
    hidden = None

    # ----------------------------------------------------
    # Iterate over training batches
    # ----------------------------------------------------
    for inputs, lengths, labels in train_loader:

        # Move tensors to the selected device
        inputs = inputs.to(device)

        lengths = lengths.to(device)

        labels = labels.to(device)

        # Reset gradients from previous iteration
        optimizer.zero_grad()

        # Detach hidden state from previous computation graph
        hidden = detach_state(hidden)

        # Forward pass through the model
        logits, features, hidden, unpacked = model(
            inputs,
            lengths,
            hidden
        )

        # ----------------------------------------------------
        # Compute loss
        # ----------------------------------------------------
        loss = criterion(
            logits,
            labels
        )

        # Backpropagation
        loss.backward()

        # ----------------------------------------------------
        # Gradient clipping to stabilize training
        # Prevents exploding gradients in RNNs
        # ----------------------------------------------------
        clip_grad_norm_(
            model.parameters(),
            max_norm=max_norm
        )

        # Update model parameters
        optimizer.step()

        # ----------------------------------------------------
        # Accumulate loss statistics
        # ----------------------------------------------------
        total_loss += (
            loss.item() * labels.size(0)
        )

        # Compute predictions
        predictions = logits.argmax(dim=1)

        # Count correct predictions
        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += labels.size(0)

        # Reset hidden state between batches
        hidden = None

    # ----------------------------------------------------
    # Compute epoch metrics
    # ----------------------------------------------------
    epoch_loss = (
        total_loss / total_samples
    )

    epoch_acc = (
        total_correct / total_samples
    )

    # Store training history
    history_loss.append(epoch_loss)

    history_acc.append(epoch_acc)

    # Print training progress
    print(
        f"Epoch {epoch+1} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_acc:.4f}"
    )


Epoch 1 | Loss: 0.7047 | Accuracy: 0.4961
Epoch 2 | Loss: 0.6729 | Accuracy: 0.5820
Epoch 3 | Loss: 0.6467 | Accuracy: 0.6289
Epoch 4 | Loss: 0.6073 | Accuracy: 0.6797
Epoch 5 | Loss: 0.5968 | Accuracy: 0.6836
